# House Prices - Preprocessing

## 1. Data Loading

This notebook builds the preprocessing pipeline for the House Prices regression task.

The goal is to handle missing values, categorical variables, numerical scaling, and target transformation in a reproducible way without data leakage.

In [33]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, ElasticNet, Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer

In [2]:
DATA_DIR = Path('../data')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')


In [3]:
X = train.drop(columns='SalePrice')
y = train['SalePrice']

print(X.shape)
print(y.shape)
print(test.shape)

(1460, 80)
(1460,)
(1459, 80)


In [4]:
y_log = np.log1p(y)

y_log.describe()

count    1460.000000
mean       12.024057
std         0.399449
min        10.460271
25%        11.775105
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64

## 2. Feature Types

Features are separated into numerical and categorical groups.

Some columns may have a numerical dtype while actually representing categories.  
For example, `MSSubClass` is an integer code describing the type of dwelling, so it should be treated as a categorical feature.

In [5]:
categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features.append("MSSubClass")
numerical_features.remove("MSSubClass")

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))
print("Total:", len(categorical_features) + len(numerical_features))

Categorical features: 44
Numerical features: 36
Total: 80


In [6]:
set(categorical_features) & set(numerical_features)

set()

## 3. Missing Values Strategy

Missing values in this dataset have different meanings.

Some missing values indicate that a property component does not exist, while others represent genuinely unknown information. These cases should be handled differently.

In [14]:
structural_cat_missing = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType",
]

structural_num_missing = [
    "GarageYrBlt",
    "MasVnrArea",
]

true_cat_missing = [
    "Electrical",
]

true_num_missing = [
    "LotFrontage",
]

regular_cat = [
    col for col in categorical_features
    if col not in structural_cat_missing
    and col not in true_cat_missing
]

regular_num = [
    col for col in numerical_features
    if col not in structural_num_missing
    and col not in true_num_missing
]

In [15]:
missing_cols = set(X.columns[X.isna().any()])

handled_missing_cols = set(
    structural_cat_missing
    + structural_num_missing
    + true_cat_missing
    + true_num_missing
)

print("Not handled:")
print(missing_cols - handled_missing_cols)

print("\nListed but not missing:")
print(handled_missing_cols - missing_cols)

Not handled:
set()

Listed but not missing:
set()


In [16]:
structural_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

true_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

structural_num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler()),
])

true_num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

regular_cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

regular_num_pipeline = Pipeline([
    ('scaler', StandardScaler()),
])

In [17]:
preprocessor = ColumnTransformer([
    ('structural_cat', structural_cat_pipeline, structural_cat_missing),
    ('true_cat', true_cat_pipeline, true_cat_missing),
    ('regular_cat', regular_cat_pipeline, regular_cat),
    ('structural_num', structural_num_pipeline, structural_num_missing),
    ('true_num', true_num_pipeline, true_num_missing),
    ('regular_num', regular_num_pipeline, regular_num),
])

In [20]:
X_transformed = preprocessor.fit_transform(X)

print(X.shape)
print(X_transformed.shape)

(1460, 80)
(1460, 317)


In [25]:
X_transformed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 116800 stored elements and shape (1460, 317)>

In [28]:
X_transformed = preprocessor.fit_transform(X)

linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    estimator=linear_model,
    X=X,
    y=y_log,
    cv=cv,
    scoring='neg_root_mean_squared_error',
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [0.13669228 0.12585663 0.23171676 0.15568899 0.11491202]
Mean RMSE: 0.15297333438596467
Std RMSE: 0.04160998300049162


In [31]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0)),
])

scores = cross_val_score(
    ridge_model,
    X,
    y_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [0.13383925 0.12748779 0.22844446 0.13268807 0.11661865]
Mean RMSE: 0.14781564686866774
Std RMSE: 0.040771714014352534


In [34]:
models = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.001, max_iter=10000),
    "elastic_net": ElasticNet(
        alpha=0.001,
        l1_ratio=0.5,
        max_iter=10000,
    ),
    "knn": KNeighborsRegressor(
        n_neighbors=5,
    ),
    "decision_tree": DecisionTreeRegressor(
        random_state=42,
    ),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
    )

    rmse = -scores

    results.append({
        "model": name,
        "mean_rmse": rmse.mean(),
        "std_rmse": rmse.std(),
    })

results = (
    pd.DataFrame(results)
    .sort_values("mean_rmse")
)

results

,model,mean_rmse,std_rmse
3,elastic_net,0.142812,0.042272
2,lasso,0.143840,0.043687
6,random_forest,0.144644,0.018973
1,ridge,0.147816,0.040772
0,linear_regression,0.152973,0.041610
4,knn,0.178279,0.014281
5,decision_tree,0.210953,0.008841
